# Initialise research documentation (run once)

Creates and seeds the three living documents that sit alongside
`decisions_log.md` in `reports/advisor_notes/`:

| Document | Question it answers | How it grows |
|---|---|---|
| `findings_log.md` | *What did we find?* | `log_finding(...)` at the end of each experiment notebook |
| `open_questions.md` | *What did we notice but park?* | `add_open_question(...)` / `resolve_open_question(...)` |
| `thesis_journal.md` | *Where is this going?* | `add_journal_entry(...)`, written by hand at session boundaries |

Together with `decisions_log.md` these are the **four pillars** of the project
record. The decisions log captures *outcomes* of thought; the findings log
captures *what we learned*; open questions capture *what we deferred*; the
journal captures *the narrative*.

This notebook is **idempotent** — re-running it will not duplicate entries, and
it does **not** touch `decisions_log.md`.


## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')

from src.utils.documentation import (
    init_documents, log_finding, add_open_question,
    resolve_open_question, add_journal_entry,
    view_findings, view_open_questions, view_journal,
)

init_documents()


Mounted at /content/drive
  created: /content/drive/MyDrive/phd_thesis/reports/advisor_notes/findings_log.md
  created: /content/drive/MyDrive/phd_thesis/reports/advisor_notes/open_questions.md
  created: /content/drive/MyDrive/phd_thesis/reports/advisor_notes/thesis_journal.md


## 2. Seed the findings log

Four findings, chronological. Each is something we *measured*, not something we
suspect — suspicions go in open questions instead.


In [2]:
log_finding(
    finding_id='F001',
    title='Synthetic pilot (v1-v4b): the contribution is localization, not detection timing',
    context='pilot experiments, prior to real data',
    what_we_did=(
        'Built controlled synthetic drift (relevance-swap and distribution-shift) and '
        'tested several explanation-space signals -- single-model SHAP rank stability, '
        'dual-model SHAP divergence (Spearman, Jaccard, KL, Wasserstein, PSI), and a '
        'prediction-disagreement baseline -- for both drift DETECTION and feature-level '
        'LOCALIZATION.'
    ),
    what_we_found=(
        '(1) Single-model SHAP fails as a leading drift indicator. '
        '(2) Dual-model SHAP detection only matches the prediction-disagreement baseline; '
        'the apparent "lead time" was a noise-scaling artifact. '
        '(3) Dual-model SHAP LOCALIZATION wins decisively: precision@K = 1.0 vs 0.46-0.75 '
        'for the permutation baseline, across both drift types. '
        '(4) Localization holds even when the two models still agree on predictions.'
    ),
    why_it_matters=(
        'Narrows the surviving novelty from "SHAP detects drift early" (falsified) to '
        '"dual-model SHAP localizes which features drifted" (supported in synthetic). '
        'This is the claim the real-data experiments must now test.'
    ),
    surprised=(
        'The leading-indicator intuition -- the original thesis hook -- was wrong. The '
        'value lives in WHICH features changed, not WHEN.'
    ),
)

log_finding(
    finding_id='F002',
    title='CICIDS2017 drift is real but moderate (notebook 05)',
    context='01_phase0_foundation/05_data_characterization',
    what_we_did=(
        'Quantified per-feature distribution shift (KS-D per feature, each day vs Monday), '
        'per-day class composition, constant/near-constant features, cross-day duplicates, '
        'and per-batch SHAP compute cost.'
    ),
    what_we_found=(
        'KS-D peaks around 0.45 (Friday) -- moderate, not categorical. The most-drifted '
        'features are packet-length statistics (fwd_packet_length_max, packet_length_std, '
        'and similar). Zero truly-constant features (Engelen cleaning is thorough); 8 '
        'near-constant flag features kept. Cross-day duplicates 0.402% overall, with '
        'Thursday-Friday anomalously high (74 vs ~30 elsewhere). TreeSHAP ~6.5s per 1000 '
        'samples, so a full streaming matrix is ~54 min per drift type -- tractable on CPU.'
    ),
    why_it_matters=(
        'Real drift is far gentler than the synthetic regime (KS approx 1.0). The synthetic '
        'precision@K = 1.0 will NOT transfer; expect substantial degradation on real data. '
        'The KS-top-K features become the first ground-truth drifted set (D008).'
    ),
)

log_finding(
    finding_id='F003',
    title='Cold-start RF collapses on post-drift attack days (notebook 06)',
    context='01_phase0_foundation/06_baseline_rf_per_day',
    what_we_did=(
        'Trained five baselines (cold_start, cold_start_balanced, cold_start_downsampled, '
        'realistic, oracle) and evaluated per-day recall, a multi-threshold sweep, and a '
        'per-attack-category breakdown.'
    ),
    what_we_found=(
        'Cold-start gets 0.981 recall on Tuesday (the brute-force attacks it trained on) '
        'and EXACTLY 0.000 on Wednesday, Thursday, and Friday. Oracle gets 0.99+ every day. '
        'Max attack probability for cold-start on Wednesday is ~0.16, so no decision '
        'threshold recovers it (recall ~0.01 even at threshold 0.01). Headroom '
        '(oracle - cold_start) on attack days is +0.749.'
    ),
    why_it_matters=(
        'Empirically validates the drift premise on real data, with large headroom for any '
        'drift-handling method to demonstrate value. The honest headline is "0.000 on '
        'Wed/Thu/Fri", not the Tuesday-inflated 0.245 mean.'
    ),
)

log_finding(
    finding_id='F004',
    title='Drift, not class imbalance, is the dominant failure mode (notebook 06, D015)',
    context='01_phase0_foundation/06_baseline_rf_per_day',
    what_we_did=(
        'Compared the imbalance-control baselines (class_weight="balanced", benign '
        'downsampling to ~100:1) and the larger-data baseline (realistic) against raw '
        'cold-start.'
    ),
    what_we_found=(
        'Imbalance handling recovers +0.001 mean recall; doubling the Tuesday training data '
        'recovers ~+0.004. Both are measurement noise. Only the oracle -- trained on the '
        'post-drift distribution -- recovers recall (0.994).'
    ),
    why_it_matters=(
        'Rules out class imbalance, training-set size, and threshold calibration as the '
        'cause. The model must LEARN post-drift attack signatures; it cannot be coaxed there '
        'by reweighting. This is classical concept drift (P(y|X) shifts across days) and it '
        'fixes cold_start as the Phase 1 reference baseline.'
    ),
)


[FINDING LOGGED] F001: Synthetic pilot (v1-v4b): the contribution is localization, not detection timing
[FINDING LOGGED] F002: CICIDS2017 drift is real but moderate (notebook 05)
[FINDING LOGGED] F003: Cold-start RF collapses on post-drift attack days (notebook 06)
[FINDING LOGGED] F004: Drift, not class imbalance, is the dominant failure mode (notebook 06, D015)


## 3. Seed the open questions

Six carried over from Phase 0, plus two new ones raised while reviewing
notebook 06 (Q007, Q008). Q007 and Q008 are load-bearing for the **notebook 07**
design — they are the reasons the Phase 1 experiment needs an `aux_only_shap`
baseline and a second, model-relative ground-truth set.


In [3]:
add_open_question(
    question_id='Q001',
    question='Why is the Thursday-Friday cross-day duplicate rate anomalously high (74 vs ~30)?',
    context='05_data_characterization',
    why_parked='Overall duplicate rate is 0.402%, below the 1% acceptability bar (D010).',
    revisit_when='If day-split leakage is suspected, or if Thursday/Friday results look anomalous in Phase 1.',
)

add_open_question(
    question_id='Q002',
    question='Should the remaining int64 feature columns be downcast (they exceed int32 range)?',
    context='03_csv_to_parquet / 06 dtype check',
    why_parked='Functionally fine for RF and TreeSHAP; only ~30 MB overhead.',
    revisit_when='If memory-bound during streaming experiments, or before any public data release.',
)

add_open_question(
    question_id='Q003',
    question='Do the near-constant flag features (urg/cwr/ece, ICMP code/type) carry rare-attack signal?',
    context='05_data_characterization, D009',
    why_parked='Kept them; the cost of keeping is low.',
    revisit_when='During feature ablation, or if SHAP flags them as important under drift.',
)

add_open_question(
    question_id='Q004',
    question='Will CSE-CIC-IDS-2018 be needed as a Phase 2 dataset?',
    context='D005 (34 GB downloaded but not converted to Parquet)',
    why_parked='UNSW-NB15 is the primary second dataset; converting 34 GB now is premature.',
    revisit_when='Phase 2 multi-dataset generalization design.',
)

add_open_question(
    question_id='Q005',
    question='Should Engelen *-Attempted rows be dropped for a robustness ablation?',
    context='D002 (currently *-Attempted counts as attack)',
    why_parked='Tangential to the drift / localization contribution.',
    revisit_when='Robustness ablation when writing the paper.',
)

add_open_question(
    question_id='Q006',
    question='How should Heartbleed (only 11 flows) be handled in per-attack analysis?',
    context='D006',
    why_parked='Effectively absent for binary classification.',
    revisit_when='If per-attack-category results need it; otherwise note as a dataset limitation.',
)

add_open_question(
    question_id='Q007',
    question='Does dual-model SHAP divergence localize better than the auxiliary-model SHAP importance ALONE?',
    context='Raised reviewing notebook 06: cold_start collapses to all-benign on Wed-Fri',
    why_parked='Requires the Phase 1 main experiment (notebook 07) to test.',
    revisit_when=(
        'Notebook 07 MUST include an aux_only_shap localization baseline. The risk: on '
        'Wed-Fri the frozen model predicts all-benign, so its SHAP may be near-zero/noisy '
        'and the divergence could reduce trivially to the auxiliary-model importance. If '
        'dual-model does NOT beat aux_only, reframe the contribution as '
        '"auxiliary-model explanation localization" rather than "dual-model divergence".'
    ),
)

add_open_question(
    question_id='Q008',
    question='Does KS-D (input-distribution drift) align with SHAP-importance drift for the ground-truth drifted set?',
    context='Raised reviewing D008 / D012',
    why_parked='Requires notebook 07 to measure both.',
    revisit_when=(
        'Notebook 07 should define a SECOND, model-relative ground truth (features the '
        'oracle relies on that cold_start does not) and report precision@K against BOTH the '
        'KS-based and the model-relative sets. If they disagree, that disagreement is itself '
        'a finding worth reporting.'
    ),
)


[QUESTION LOGGED] Q001: Why is the Thursday-Friday cross-day duplicate rate anomalously high (...
[QUESTION LOGGED] Q002: Should the remaining int64 feature columns be downcast (they exceed in...
[QUESTION LOGGED] Q003: Do the near-constant flag features (urg/cwr/ece, ICMP code/type) carry...
[QUESTION LOGGED] Q004: Will CSE-CIC-IDS-2018 be needed as a Phase 2 dataset?...
[QUESTION LOGGED] Q005: Should Engelen *-Attempted rows be dropped for a robustness ablation?...
[QUESTION LOGGED] Q006: How should Heartbleed (only 11 flows) be handled in per-attack analysi...
[QUESTION LOGGED] Q007: Does dual-model SHAP divergence localize better than the auxiliary-mod...
[QUESTION LOGGED] Q008: Does KS-D (input-distribution drift) align with SHAP-importance drift ...


## 4. Seed the journal

Two entries to anchor the narrative. From here on, journal entries are written
**by hand** at the end of a work session — they are reflection, not generated
output. Omit `entry_id` for hand-written entries (they always append).


In [4]:
add_journal_entry(
    entry_id='J001',
    title='The pivot: from adaptive feature prioritization to explanation-space dynamics',
    body=(
        'The original thesis hook was "feedback-driven adaptive feature prioritization for '
        'IDS." A 27-paper literature pass showed that frame is already occupied from several '
        'angles (IGPC-MSOS dual-trigger mode-switching; Soltani multi-agent continual '
        'learning; Hossain RL with experience replay; Ahmed and Nugraha SHAP-based feature '
        'selection with retraining). What survived was narrower and sharper: explanation '
        'drift as the signal. The supervisor approved the pivot. The working title is now '
        '"Explanation-Space Dynamics under Concept Drift in IDS." The single sharpest '
        'distinction against the closest competitor: IGPC-MSOS handles feature-SPACE drift '
        '(features appearing or disappearing) via FA-OSELM, but not feature-IMPORTANCE '
        'drift (rank reordering with a stable feature space) -- which is what this work '
        'probes.'
    ),
)

add_journal_entry(
    entry_id='J002',
    title='End of Phase 0: premise confirmed, dual-model framing faces its real test',
    body=(
        'Infrastructure is done -- Parquet pipeline, unified loaders, paper-quality '
        'plotting, decisions log. The drift premise is now confirmed on real data: '
        'cold-start RF collapses to 0.000 recall on Wed/Thu/Fri while the oracle reaches '
        '0.99+, and class imbalance is ruled out as the cause (F003, F004).\n\n'
        'Two things temper the optimism. First, synthetic precision@K = 1.0 will not '
        'survive real, gentler drift -- the pre-registered STRONG/MODEST/WEAK criteria '
        '(D012) are the guardrail against reading too much into whatever number comes back. '
        'Second, and more pointed: because the frozen model collapses to all-benign on the '
        'post-drift days, the "dual-model" divergence may reduce to "auxiliary-model '
        'importance" with the frozen model contributing nothing (Q007). Notebook 07 is '
        'designed to confront this directly with an aux_only baseline. If the dual-model '
        'framing does not earn its keep there, the contribution narrows honestly rather '
        'than collapsing. Next step: build notebook 07.'
    ),
)


[JOURNAL ENTRY] The pivot: from adaptive feature prioritization to explanation-space dynamics
[JOURNAL ENTRY] End of Phase 0: premise confirmed, dual-model framing faces its real test


## 5. Verify

In [5]:
print('=' * 70)
print('FINDINGS LOG')
print('=' * 70)
view_findings()

print('=' * 70)
print('OPEN QUESTIONS')
print('=' * 70)
view_open_questions()

print('=' * 70)
print('THESIS JOURNAL')
print('=' * 70)
view_journal()


FINDINGS LOG
# Findings Log

Chronological record of empirical discoveries. Append-only. Each
entry records what we did, what we found (with numbers), why it
matters, and what (if anything) surprised us.

Seeded and appended via `src/utils/documentation.py::log_finding`.

## F001 — Synthetic pilot (v1-v4b): the contribution is localization, not detection timing
*Logged 2026-06-04 | context: `pilot experiments, prior to real data`*

**What we did.** Built controlled synthetic drift (relevance-swap and distribution-shift) and tested several explanation-space signals -- single-model SHAP rank stability, dual-model SHAP divergence (Spearman, Jaccard, KL, Wasserstein, PSI), and a prediction-disagreement baseline -- for both drift DETECTION and feature-level LOCALIZATION.

**What we found.** (1) Single-model SHAP fails as a leading drift indicator. (2) Dual-model SHAP detection only matches the prediction-disagreement baseline; the apparent "lead time" was a noise-scaling artifact. (3) Dual-

## 6. How to use these going forward

**Findings** — end every experiment notebook (07 onward) with one or more
`log_finding(...)` calls, the same way you end with `log_decision(...)`. A
finding is something you measured.

**Open questions** — call `add_open_question(...)` the moment you notice
something you are choosing not to chase right now. When you later answer one,
call:

```python
resolve_open_question('Q007', 'Dual-model beat aux_only by +0.18 precision@K on Wed; dual framing holds. See F0xx.')
```

This flips the Status line in place and prints the change. Everything else is
append-only.

**Journal** — write a short entry by hand whenever the direction shifts or a
session ends with something worth remembering:

```python
add_journal_entry(
    title='Notebook 07 first results',
    body='...what happened, what it means, what worries me, what is next...',
)
```

No `entry_id` for hand-written entries — they always append.

---

*Separate reminder, unrelated to these docs:* confirm the `D015` entry in
`decisions_log.md` reads as **drift-dominant** (not the old "mixed" text). If it
still says mixed, delete that block and re-run cell 12 of notebook 06.
